# Extracting New Product Announcements from SEC Form 8-K Filings with LLMs

**AI for Finance, UCF MS in Financial Technology**

This notebook is a thin driver over two scripts in this repo:

1. `fetch_filings.py` pulls 8-K press-release exhibits from EDGAR full-text search.
2. `extract_products.py` runs spaCy NER plus a local LLM (Ollama) over each filing and writes a CSV.

Run the cells top to bottom from the repo root (`jupyter notebook` launched in the project folder). Settings live in `config.py`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import config
print("LLM backend:", config.LLM_BACKEND, "| model:", config.OLLAMA_MODEL)
print("User-Agent :", config.SEC_USER_AGENT)

## 1. Collect filings from EDGAR

In [ ]:
import fetch_filings
fetch_filings.main(max_filings=config.MAX_FILINGS)

## 2. Peek at one raw filing

In [ ]:
import json
sample = sorted(config.RAW_DIR.glob('*.json'))[0]
rec = json.loads(sample.read_text(encoding='utf-8'))
print(rec['company_name'], rec['ticker'], rec['filing_date'])
print(rec['text'][:1500])

## 3. NER candidates (spaCy) for that filing

In [ ]:
import extract_products as ep
orgs, prods = ep.ner_candidates(ep.focus_text(rec['text'], config.MAX_CONTEXT_CHARS))
print('ORG     :', orgs)
print('PRODUCT :', prods)

## 4. Run the LLM over every filing and write the CSV

In [ ]:
ep.main(limit=0, backend=config.LLM_BACKEND, out_name='extraction_full.csv')

## 5. Results

In [ ]:
import pandas as pd
df = pd.read_csv(config.OUTPUT_DIR / 'new_products.csv')
print(len(df), 'product rows from', df.stock_name.nunique(), 'companies')
print('max description length:', df.product_description.str.len().max())
df.head(20)

In [ ]:
# Pipe-separated view, as in the assignment prompt
print(df.to_csv(sep='|', index=False)[:3000])